In [2]:
# Prepara reanalise ERA5 para input do SWAN
# Henrique Pereira - 14/03/2025
# ~ AtmosMarine ~

In [3]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cmocean

In [4]:
# windwave

ds_wind = xr.open_dataset('/mnt/c/Users/henri/OneDrive - atmosmarine.com/database/ERA5/Brasil/ERA5_Brasil_202411/data_stream-oper_stepType-instant.nc')
ds_wave = xr.open_dataset('/mnt/c/Users/henri/OneDrive - atmosmarine.com/database/ERA5/Brasil/ERA5_Brasil_202411/data_stream-wave_stepType-instant.nc')

ds_wind = ds_wind.sel(latitude=slice(-22.0, -24.0), longitude=slice(-45.0, -41.0))
ds_wave = ds_wave.sel(latitude=slice(-22.0, -24.0), longitude=slice(-45.0, -41.0))

# Reindexando ds1 para a grade de ds2 - tem que ser da de mais resolucao para a de menor
ds_wind = ds_wind.reindex_like(ds_wave, method='nearest') # certo - 
# ds_wave = ds_wave.reindex_like(ds_wind, method='nearest') # errado

ds_windwave = xr.merge([ds_wind, ds_wave])

ds_windwave = ds_windwave.drop_vars(['expver', 'number'])

ds_windwave = ds_windwave.rename_dims({'valid_time': 'time'})

# Para o SWAN
ds_windwave1 = ds_windwave[['u10', 'v10', 'swh', 'mwd', 'mwp']]
# Inverter a ordem da latitude e Reindexar as variáveis para manter a consistência
# ds_windwave1['latitude'] = ds_windwave1['latitude'][::-1]
ds_windwave1 = ds_windwave1.reindex(latitude=ds_windwave1['latitude'])
ds_windwave1 = ds_windwave1.rename_vars({'valid_time': 'time'})
# ds_windwave1
ds_windwave1.to_netcdf('RJ/NCfiles/windwave_rj_era5_050_202411.nc')

In [5]:
ds_windwave1

<xarray.Dataset> Size: 654kB
Dimensions:    (time: 720, latitude: 5, longitude: 9)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2024-11-01 ... 2024-11-30T23:00:00
  * latitude   (latitude) float64 40B -22.0 -22.5 -23.0 -23.5 -24.0
  * longitude  (longitude) float64 72B -45.0 -44.5 -44.0 ... -42.0 -41.5 -41.0
Data variables:
    u10        (time, latitude, longitude) float32 130kB ...
    v10        (time, latitude, longitude) float32 130kB ...
    swh        (time, latitude, longitude) float32 130kB ...
    mwd        (time, latitude, longitude) float32 130kB ...
    mwp        (time, latitude, longitude) float32 130kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-02-23T01:34 GRIB to CDM+CF via cfgrib-0.9.1...